In [1]:
import sys; sys.path.append("..")
from pyspark.sql import functions as F, types as T, Window
from src.spark import get_spark
from src.paths import RAW, BRONZE, SILVER, GOLD, EXTERNAL
from src.schemas import SCHEMAS

spark = get_spark()
spark

26/09/16 09:53:00 WARN Utils: Your hostname, Leventes-MacBook-Air-M4.local resolves to a loopback address: 127.0.0.1; using 172.20.10.14 instead (on interface en0)
26/09/16 09:53:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/16 09:53:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
bronze = {}
for name, (csv, schema) in SCHEMAS.items():
    df = spark.read.csv(str(RAW / csv), header=True, schema=schema)
    df.write.mode("overwrite").parquet(str(BRONZE / name))
    bronze[name] = spark.read.parquet(str(BRONZE / name))
    print(f"{name:13s} {bronze[name].count():>9,d} rows")

sales         3,000,888 rows
stores               54 rows
oil               1,218 rows
holidays            350 rows
transactions     83,488 rows


In [3]:
bronze["sales"].printSchema()

root
 |-- id: long (nullable = true)
 |-- date: date (nullable = true)
 |-- store_nbr: integer (nullable = true)
 |-- family: string (nullable = true)
 |-- sales: double (nullable = true)
 |-- onpromotion: integer (nullable = true)

